In [8]:
# ============================================
# BLOCK 1: IMPORT LIBRARIES
# RETENTION INTELLIGENCE ENGINE
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Load final model
final_xgboost_churn_model = joblib.load(
    "models/final_xgboost_churn_model.pkl"
)

# Load processed test data
X_test_processed = joblib.load(
    "models/X_test_processed.pkl"
)

# Load test labels
y_test = joblib.load(
    "models/y_test.pkl"
)

# Load feature names
feature_names = joblib.load(
    "models/feature_names.pkl"
)

print("Model loaded:", type(final_xgboost_churn_model))
print("X_test_processed:", X_test_processed.shape)
print("y_test:", np.asarray(y_test).shape)
print("Features:", len(feature_names))

Model loaded: <class 'xgboost.sklearn.XGBClassifier'>
X_test_processed: (1405, 45)
y_test: (1405,)
Features: 45


In [11]:
# ============================================
# RETENTION INTELLIGENCE ENGINE
# GENERATE CHURN PREDICTIONS
# ============================================

# Generate predicted churn classes
customer_churn_prediction = final_xgboost_churn_model.predict(
    X_test_processed
)

# Generate churn probabilities
# Extract probability of Churn = 1
customer_churn_probability = final_xgboost_churn_model.predict_proba(
    X_test_processed
)[:, 1]

# Create retention DataFrame
retention_df = pd.DataFrame({
    "Actual_Churn": np.asarray(y_test),
    "Churn_Prediction": customer_churn_prediction,
    "Churn_Probability": customer_churn_probability
})

# Display first 10 rows
print("First 10 Customer Retention Records:")
print("=" * 60)
display(retention_df.head(10))

# Display churn probability statistics
print("\nChurn Probability Statistics:")
print("=" * 60)

print(f"Minimum Churn Probability: {customer_churn_probability.min():.4f}")
print(f"Maximum Churn Probability: {customer_churn_probability.max():.4f}")
print(f"Average Churn Probability: {customer_churn_probability.mean():.4f}")

First 10 Customer Retention Records:


,Actual_Churn,Churn_Prediction,Churn_Probability
0,0,1,0.856967
1,1,0,0.354820
2,1,1,0.602191
3,0,0,0.084179
4,0,0,0.171892
5,0,0,0.456559
6,0,0,0.111514
7,1,1,0.908862
8,0,0,0.008605
9,0,0,0.008524



Churn Probability Statistics:
Minimum Churn Probability: 0.0010
Maximum Churn Probability: 0.9599
Average Churn Probability: 0.2966


In [12]:
# ============================================
# CUSTOMER RISK CLASSIFICATION
# ============================================

# Define risk levels based on churn probability
conditions = [
    retention_df["Churn_Probability"] < 0.30,
    (retention_df["Churn_Probability"] >= 0.30) &
    (retention_df["Churn_Probability"] < 0.60),
    (retention_df["Churn_Probability"] >= 0.60) &
    (retention_df["Churn_Probability"] < 0.80),
    retention_df["Churn_Probability"] >= 0.80
]

risk_labels = [
    "Low Risk",
    "Medium Risk",
    "High Risk",
    "Critical Risk"
]

# Create Risk_Level column
retention_df["Risk_Level"] = np.select(
    conditions,
    risk_labels,
    default="Unknown"
)

# ============================================
# 1. DISPLAY FIRST 10 CUSTOMERS
# ============================================

print("First 10 Customers - Risk Classification:")
print("=" * 60)

display(
    retention_df[
        ["Churn_Probability", "Risk_Level"]
    ].head(10)
)

# ============================================
# 2. CUSTOMER COUNT BY RISK LEVEL
# ============================================

risk_counts = retention_df["Risk_Level"].value_counts()

print("\nCustomer Count by Risk Level:")
print("=" * 60)

print(risk_counts)

# ============================================
# 3. CUSTOMER PERCENTAGE BY RISK LEVEL
# ============================================

risk_percentages = (
    retention_df["Risk_Level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nCustomer Percentage by Risk Level:")
print("=" * 60)

print(risk_percentages)

First 10 Customers - Risk Classification:


,Churn_Probability,Risk_Level
0,0.856967,Critical Risk
1,0.354820,Medium Risk
2,0.602191,High Risk
3,0.084179,Low Risk
4,0.171892,Low Risk
5,0.456559,Medium Risk
6,0.111514,Low Risk
7,0.908862,Critical Risk
8,0.008605,Low Risk
9,0.008524,Low Risk



Customer Count by Risk Level:
Risk_Level
Low Risk         853
Medium Risk      268
High Risk        168
Critical Risk    116
Name: count, dtype: int64

Customer Percentage by Risk Level:
Risk_Level
Low Risk         60.71
Medium Risk      19.07
High Risk        11.96
Critical Risk     8.26
Name: proportion, dtype: float64


In [14]:
import joblib

X_test = joblib.load("X_test.pkl")

print(X_test.shape)
print(X_test["MonthlyCharges"].head())

(1405, 19)
5627    74.85
6126    95.90
2361    45.95
2201    85.85
832     74.10
Name: MonthlyCharges, dtype: float64


In [15]:
# ============================================
# RETENTION PRIORITY SYSTEM
# ============================================

# Copy MonthlyCharges from original feature-engineered test data
retention_df["MonthlyCharges"] = X_test["MonthlyCharges"].values

# Calculate MonthlyCharges percentiles
monthly_33rd = X_test["MonthlyCharges"].quantile(0.33)
monthly_66th = X_test["MonthlyCharges"].quantile(0.66)

# ============================================
# CUSTOMER VALUE SCORE
# ============================================

retention_df["Customer_Value_Score"] = np.select(
    [
        retention_df["MonthlyCharges"] < monthly_33rd,
        (retention_df["MonthlyCharges"] >= monthly_33rd) &
        (retention_df["MonthlyCharges"] <= monthly_66th),
        retention_df["MonthlyCharges"] > monthly_66th
    ],
    [
        "Low Value",
        "Medium Value",
        "High Value"
    ],
    default="Unknown"
)

# ============================================
# RETENTION PRIORITY
# ============================================

retention_df["Retention_Priority"] = np.select(
    [
        (retention_df["Risk_Level"] == "Critical Risk") &
        (retention_df["Customer_Value_Score"] == "High Value"),

        (retention_df["Risk_Level"] == "High Risk") &
        (retention_df["Customer_Value_Score"] == "High Value"),

        (retention_df["Risk_Level"] == "Critical Risk") &
        (retention_df["Customer_Value_Score"] == "Medium Value"),

        (retention_df["Risk_Level"] == "High Risk") &
        (retention_df["Customer_Value_Score"] == "Medium Value"),

        (retention_df["Risk_Level"] == "Medium Risk") &
        (retention_df["Customer_Value_Score"] == "High Value")
    ],
    [
        "Critical",
        "High",
        "High",
        "Medium",
        "Medium"
    ],
    default="Low"
)

# ============================================
# SORT RETENTION PRIORITY
# ============================================

priority_order = {
    "Critical": 0,
    "High": 1,
    "Medium": 2,
    "Low": 3
}

retention_df["Priority_Order"] = retention_df["Retention_Priority"].map(
    priority_order
)

# Sort by Retention Priority and then Churn Probability descending
retention_priority_df = (
    retention_df
    .sort_values(
        by=["Priority_Order", "Churn_Probability"],
        ascending=[True, False]
    )
    .drop(columns=["Priority_Order"])
)

# ============================================
# DISPLAY TOP 20 PRIORITY CUSTOMERS
# ============================================

print("Top 20 Customers by Retention Priority:")
print("=" * 80)

display(
    retention_priority_df[
        [
            "MonthlyCharges",
            "Churn_Probability",
            "Risk_Level",
            "Customer_Value_Score",
            "Retention_Priority"
        ]
    ].head(20)
)

Top 20 Customers by Retention Priority:


,MonthlyCharges,Churn_Probability,Risk_Level,Customer_Value_Score,Retention_Priority
830,100.80,0.959942,Critical Risk,High Value,Critical
1039,98.25,0.942054,Critical Risk,High Value,Critical
761,86.05,0.941328,Critical Risk,High Value,Critical
216,86.60,0.935812,Critical Risk,High Value,Critical
467,93.85,0.933201,Critical Risk,High Value,Critical
709,94.00,0.933201,Critical Risk,High Value,Critical
1261,95.15,0.930390,Critical Risk,High Value,Critical
1311,100.15,0.927512,Critical Risk,High Value,Critical
89,89.55,0.923967,Critical Risk,High Value,Critical
1321,100.50,0.922013,Critical Risk,High Value,Critical


In [16]:
# ============================================
# RETENTION INTELLIGENCE BUSINESS SUMMARY
# ============================================

# 1. Number of customers in each Retention_Priority category
priority_counts = retention_df["Retention_Priority"].value_counts()

print("============================================")
print("CUSTOMER COUNT BY RETENTION PRIORITY")
print("============================================")
print(priority_counts)


# 2. Percentage of customers in each Retention_Priority category
priority_percentages = (
    retention_df["Retention_Priority"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\n============================================")
print("CUSTOMER PERCENTAGE BY RETENTION PRIORITY")
print("============================================")
print(priority_percentages)


# 3. Cross-tabulation of Risk_Level vs Customer_Value_Score
risk_value_crosstab = pd.crosstab(
    retention_df["Risk_Level"],
    retention_df["Customer_Value_Score"]
)

print("\n============================================")
print("RISK LEVEL VS CUSTOMER VALUE SCORE")
print("============================================")
display(risk_value_crosstab)


# 4. Cross-tabulation of Risk_Level vs Retention_Priority
risk_priority_crosstab = pd.crosstab(
    retention_df["Risk_Level"],
    retention_df["Retention_Priority"]
)

print("\n============================================")
print("RISK LEVEL VS RETENTION PRIORITY")
print("============================================")
display(risk_priority_crosstab)

CUSTOMER COUNT BY RETENTION PRIORITY
Retention_Priority
Low         1046
Medium       187
High         109
Critical      63
Name: count, dtype: int64

CUSTOMER PERCENTAGE BY RETENTION PRIORITY
Retention_Priority
Low         74.45
Medium      13.31
High         7.76
Critical     4.48
Name: proportion, dtype: float64

RISK LEVEL VS CUSTOMER VALUE SCORE


Customer_Value_Score,High Value,Low Value,Medium Value
Risk_Level,,,
Critical Risk,63,6,47
High Risk,62,36,70
Low Risk,236,345,272
Medium Risk,117,77,74



RISK LEVEL VS RETENTION PRIORITY


Retention_Priority,Critical,High,Low,Medium
Risk_Level,,,,
Critical Risk,63,47,6,0
High Risk,0,62,36,70
Low Risk,0,0,853,0
Medium Risk,0,0,151,117


In [19]:
import joblib
import numpy as np

shap_values = joblib.load(
    "models/shap_values.pkl"
)

print("SHAP values loaded successfully!")
print("SHAP values shape:", np.asarray(shap_values).shape)

SHAP values loaded successfully!
SHAP values shape: (1405, 45)


In [20]:
# ============================================
# CUSTOMER-LEVEL SHAP RETENTION DRIVERS
# ============================================

# Convert SHAP values to NumPy array if needed
shap_array = np.asarray(shap_values)

# Handle possible 3D SHAP output formats
if shap_array.ndim == 3:
    shap_array = shap_array[:, :, 1]

# Ensure SHAP rows match customer rows
assert shap_array.shape[0] == len(X_test_processed), \
    "SHAP values and X_test_processed have different numbers of customers."

# Ensure SHAP features match feature names
assert shap_array.shape[1] == len(feature_names), \
    "SHAP values and feature_names have different numbers of features."

# Ensure retention_df is aligned with X_test_processed
assert len(retention_df) == len(X_test_processed), \
    "retention_df and X_test_processed have different numbers of customers."

# Create columns for every customer
top_churn_drivers = []
top_retention_drivers = []

for i in range(len(X_test_processed)):

    # SHAP values for current customer
    customer_shap = shap_array[i]

    # Positive SHAP values -> push prediction toward churn
    positive_indices = np.where(customer_shap > 0)[0]

    # Sort positive SHAP values from highest to lowest
    positive_indices = positive_indices[
        np.argsort(customer_shap[positive_indices])[::-1]
    ]

    # Select top 3 churn drivers
    top_positive_features = [
        feature_names[idx]
        for idx in positive_indices[:3]
    ]

    # Negative SHAP values -> push prediction toward non-churn
    negative_indices = np.where(customer_shap < 0)[0]

    # Sort negative SHAP values by strongest negative contribution
    negative_indices = negative_indices[
        np.argsort(customer_shap[negative_indices])
    ]

    # Select top 3 retention drivers
    top_negative_features = [
        feature_names[idx]
        for idx in negative_indices[:3]
    ]

    # Store as comma-separated strings
    top_churn_drivers.append(
        ", ".join(top_positive_features)
    )

    top_retention_drivers.append(
        ", ".join(top_negative_features)
    )


# Add SHAP driver columns to retention_df
retention_df["Top_Churn_Drivers"] = top_churn_drivers
retention_df["Top_Retention_Drivers"] = top_retention_drivers


# ============================================
# DISPLAY FIRST 10 CUSTOMERS
# ============================================

display(
    retention_df[
        [
            "Churn_Probability",
            "Risk_Level",
            "Retention_Priority",
            "Top_Churn_Drivers",
            "Top_Retention_Drivers"
        ]
    ].head(10)
)

,Churn_Probability,Risk_Level,Retention_Priority,Top_Churn_Drivers,Top_Retention_Drivers
0,0.856967,Critical Risk,High,"categorical__Contract_Month-to-month, numerica...","categorical__DeviceProtection_Yes, categorical..."
1,0.354820,Medium Risk,Medium,"categorical__Contract_Month-to-month, categori...","numerical__tenure, categorical__PaymentMethod_..."
2,0.602191,High Risk,Low,"numerical__tenure, categorical__Contract_Month...","categorical__gender_Male, categorical__Interne..."
3,0.084179,Low Risk,Low,"categorical__gender_Female, categorical__Paper...","categorical__Contract_Month-to-month, categori..."
4,0.171892,Low Risk,Low,"categorical__gender_Female, categorical__Paper...","categorical__Contract_Month-to-month, categori..."
5,0.456559,Medium Risk,Medium,"numerical__MonthlyCharges, categorical__gender...","categorical__Contract_Month-to-month, categori..."
6,0.111514,Low Risk,Low,"categorical__Contract_Month-to-month, categori...","categorical__OnlineSecurity_No, categorical__P..."
7,0.908862,Critical Risk,Critical,"numerical__tenure, categorical__Contract_Month...","categorical__Dependents_No, categorical__Partn..."
8,0.008605,Low Risk,Low,"categorical__Partner_No, categorical__Paperles...","categorical__Contract_Month-to-month, categori..."
9,0.008524,Low Risk,Low,"categorical__PaperlessBilling_No, categorical_...","categorical__Contract_Month-to-month, numerica..."


In [21]:
# ============================================
# RETENTION ACTION RECOMMENDATION ENGINE
# ============================================

def generate_retention_action(row):
    risk = str(row["Risk_Level"]).strip()
    value = str(row["Customer_Value_Score"]).strip()
    churn_drivers = str(row.get("Top_Churn_Drivers", "")).strip()

    # Base recommendation based on risk and customer value
    if risk == "Critical Risk" and value == "High Value":
        action = (
            "Immediate personalized retention intervention: "
            "offer a targeted discount, contract upgrade, or dedicated support."
        )

    elif risk == "Critical Risk" and value in ["Medium Value", "Low Value"]:
        action = (
            "Launch a low-cost automated retention campaign or "
            "send a personalized retention offer."
        )

    elif risk == "High Risk" and value == "High Value":
        action = (
            "Proactive customer success outreach with a personalized "
            "retention offer and dedicated assistance."
        )

    elif risk == "High Risk" and value in ["Medium Value", "Low Value"]:
        action = (
            "Launch a targeted email/SMS campaign with relevant "
            "service incentives or promotional benefits."
        )

    elif risk == "Medium Risk":
        action = (
            "Run an engagement campaign with loyalty benefits, "
            "personalized communication, or a customer satisfaction survey."
        )

    else:
        action = (
            "Continue normal customer engagement and loyalty communication "
            "to maintain customer satisfaction."
        )

    # Add churn-driver context when available
    if churn_drivers and churn_drivers.lower() not in ["nan", "none", ""]:
        action += (
            f" Prioritize addressing the key churn drivers identified: "
            f"{churn_drivers}."
        )

    return action


# Apply the retention action function to every customer
retention_df["Recommended_Action"] = retention_df.apply(
    generate_retention_action,
    axis=1
)


# ============================================
# DISPLAY TOP 10 HIGHEST-PRIORITY CUSTOMERS
# ============================================

priority_order = {
    "Critical": 0,
    "High": 1,
    "Medium": 2,
    "Low": 3
}

top_priority_customers = (
    retention_df.assign(
        Priority_Order=retention_df["Retention_Priority"]
        .map(priority_order)
        .fillna(99)
    )
    .sort_values(
        by=["Priority_Order", "Churn_Probability"],
        ascending=[True, False]
    )
    .head(10)
)

display(
    top_priority_customers[
        [
            "Churn_Probability",
            "Risk_Level",
            "Retention_Priority",
            "Top_Churn_Drivers",
            "Recommended_Action"
        ]
    ]
)

,Churn_Probability,Risk_Level,Retention_Priority,Top_Churn_Drivers,Recommended_Action
830,0.959942,Critical Risk,Critical,"numerical__tenure, categorical__Contract_Month...",Immediate personalized retention intervention:...
1039,0.942054,Critical Risk,Critical,"numerical__MonthlyCharges, numerical__tenure, ...",Immediate personalized retention intervention:...
761,0.941328,Critical Risk,Critical,"numerical__tenure, categorical__Contract_Month...",Immediate personalized retention intervention:...
216,0.935812,Critical Risk,Critical,"numerical__tenure, categorical__Contract_Month...",Immediate personalized retention intervention:...
467,0.933201,Critical Risk,Critical,"categorical__Contract_Month-to-month, numerica...",Immediate personalized retention intervention:...
709,0.933201,Critical Risk,Critical,"categorical__Contract_Month-to-month, numerica...",Immediate personalized retention intervention:...
1261,0.930390,Critical Risk,Critical,"categorical__Contract_Month-to-month, numerica...",Immediate personalized retention intervention:...
1311,0.927512,Critical Risk,Critical,"numerical__MonthlyCharges, numerical__tenure, ...",Immediate personalized retention intervention:...
89,0.923967,Critical Risk,Critical,"numerical__tenure, categorical__Contract_Month...",Immediate personalized retention intervention:...
1321,0.922013,Critical Risk,Critical,"categorical__Contract_Month-to-month, numerica...",Immediate personalized retention intervention:...


In [22]:
# ============================================
# FINAL CUSTOMER RETENTION REPORT
# ============================================

# Create Customer_Index based on the original test data index
customer_retention_report = retention_df.copy()

customer_retention_report["Customer_Index"] = X_test.index

# Define priority sorting order
priority_order = {
    "Critical": 0,
    "High": 1,
    "Medium": 2,
    "Low": 3
}

# Create temporary sorting column
customer_retention_report["Priority_Order"] = (
    customer_retention_report["Retention_Priority"]
    .map(priority_order)
)

# Sort by:
# 1. Retention Priority: Critical → High → Medium → Low
# 2. Churn Probability: Descending
customer_retention_report = (
    customer_retention_report
    .sort_values(
        by=["Priority_Order", "Churn_Probability"],
        ascending=[True, False]
    )
    .drop(columns=["Priority_Order"])
    .reset_index(drop=True)
)

# Select final report columns
customer_retention_report = customer_retention_report[
    [
        "Customer_Index",
        "Churn_Probability",
        "Risk_Level",
        "MonthlyCharges",
        "Customer_Value_Score",
        "Retention_Priority",
        "Top_Churn_Drivers",
        "Top_Retention_Drivers",
        "Recommended_Action"
    ]
]

# Display first 20 customers
display(customer_retention_report.head(20))

# Print total number of customers
print("Total Number of Customers:", len(customer_retention_report))

# Print number of Critical and High priority customers
critical_count = (
    customer_retention_report["Retention_Priority"] == "Critical"
).sum()

high_count = (
    customer_retention_report["Retention_Priority"] == "High"
).sum()

print("Critical Priority Customers:", critical_count)
print("High Priority Customers:", high_count)

,Customer_Index,Churn_Probability,Risk_Level,MonthlyCharges,Customer_Value_Score,Retention_Priority,Top_Churn_Drivers,Top_Retention_Drivers,Recommended_Action
0,2208,0.959942,Critical Risk,100.80,High Value,Critical,"numerical__tenure, categorical__Contract_Month...","categorical__gender_Male, categorical__Partner...",Immediate personalized retention intervention:...
1,6215,0.942054,Critical Risk,98.25,High Value,Critical,"numerical__MonthlyCharges, numerical__tenure, ...","categorical__gender_Male, categorical__Partner...",Immediate personalized retention intervention:...
2,2745,0.941328,Critical Risk,86.05,High Value,Critical,"numerical__tenure, categorical__Contract_Month...","categorical__Partner_No, categorical__DevicePr...",Immediate personalized retention intervention:...
3,983,0.935812,Critical Risk,86.60,High Value,Critical,"numerical__tenure, categorical__Contract_Month...","categorical__Partner_Yes, categorical__DeviceP...",Immediate personalized retention intervention:...
4,3749,0.933201,Critical Risk,93.85,High Value,Critical,"categorical__Contract_Month-to-month, numerica...","categorical__Partner_Yes, categorical__DeviceP...",Immediate personalized retention intervention:...
5,6368,0.933201,Critical Risk,94.00,High Value,Critical,"categorical__Contract_Month-to-month, numerica...","categorical__Partner_Yes, categorical__DeviceP...",Immediate personalized retention intervention:...
6,1148,0.930390,Critical Risk,95.15,High Value,Critical,"categorical__Contract_Month-to-month, numerica...","categorical__gender_Male, categorical__Partner...",Immediate personalized retention intervention:...
7,2280,0.927512,Critical Risk,100.15,High Value,Critical,"numerical__MonthlyCharges, numerical__tenure, ...","categorical__gender_Male, categorical__TechSup...",Immediate personalized retention intervention:...
8,642,0.923967,Critical Risk,89.55,High Value,Critical,"numerical__tenure, categorical__Contract_Month...","categorical__MultipleLines_No, categorical__Pa...",Immediate personalized retention intervention:...
9,6232,0.922013,Critical Risk,100.50,High Value,Critical,"categorical__Contract_Month-to-month, numerica...","categorical__gender_Male, categorical__Partner...",Immediate personalized retention intervention:...


Total Number of Customers: 1405
Critical Priority Customers: 63
High Priority Customers: 109


In [23]:
import os

# Create output directory if it does not exist
os.makedirs("outputs", exist_ok=True)

# Define output file path
output_path = "outputs/customer_retention_report.csv"

# Export the retention report
customer_retention_report.to_csv(
    output_path,
    index=False
)

# Verify that the file exists
if os.path.exists(output_path):
    print("File saved successfully!")
    print("Saved file path:", output_path)
    print("Number of rows exported:", len(customer_retention_report))
else:
    print("Error: File was not saved.")

File saved successfully!
Saved file path: outputs/customer_retention_report.csv
Number of rows exported: 1405
